In [1]:
import numpy as np
import pandas as pd
import odl
import matplotlib.pyplot as plt
import sys
from functools import partial
sys.path.append("../..")

In [2]:
from src.siac_fourier import siac_filter_odl

from src.tomo import(
    make_space_2d, 
    shepp_logan_2d, 
    parallel_geom_2d, 
    ray_transform_2d, 
    reconstruct_fbp, 
    add_relative_gaussian_noise
)

from src.experiments.methods import(
    run_post_recon_dg_siac, 
    run_post_recon_fourier_siac, 
    run_pre_recon_siac_detector
)
from src.experiments.monte_carlo import run_monte_carlo_study

from src.metrics import(
    compute_metrics, 
    summarize_mc_results, 
    select_best_by_noise
    )

from src.plotting_helpers import (
    plot_img, 
    plot_img_zoom, 
    save_image_w_zoom, 
    plot_mc_metric
)


### Setup the space using ODL

In [3]:
### Create the reconstruction space and phantom ###

xmin, xmax, ymin, ymax = -20, 20, -20, 20
Nx, Ny = 100, 100

space = make_space_2d(Nx=Nx, Ny=Ny, domain=[xmin, xmax, ymin, ymax])
phantom = shepp_logan_2d(space)
phantom_np = phantom.asarray()

# Full angular coverage (mimic CT)
angular_coverage=(-90,90)
step = 3                    # angular resolution (3 degrees per step)

# detector half-width should be at least the half-diagonal of the reconstruction box
r = np.sqrt((0.5*(xmax - xmin))**2 + (0.5*(ymax - ymin))**2)
det_range = (-r, r)

det_count = int(np.ceil(1.5 * np.sqrt(Nx**2 + Ny**2)))

geom = parallel_geom_2d(angular_coverage=angular_coverage, step=step, 
                        det_range=det_range, det_count=det_count)
A = ray_transform_2d(space, geom)
data_space = A.range

sinogram = A(phantom)



/home/ahopkins/KTH_TTMAM/MEX/odl_xray/venv/lib/python3.10/site-packages/odl/util/utility.py:1398: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_requirements


### Setup helper functions for Monte Carlo simulation

In [4]:
def method_dg(input_data, xlim, ylim, p, moments, BSorder):
    return run_post_recon_dg_siac(
        recon=input_data,
        xlim=xlim,
        ylim=ylim,
        p=p,
        moments=moments,
        BSorder=BSorder
        )

def method_fourier(input_data, dy, dx, moments, BSorder):
    return run_post_recon_fourier_siac(
        recon=input_data, 
        dy=dy, 
        dx=dx, 
        moments=moments, 
        BSorder=BSorder
        )

    
def method_detector(input_data, A, d_det, moments, BSorder, pad_mode="reflect"):
    return run_pre_recon_siac_detector(
        sinogram=input_data,
        A=A,
        d_det=d_det,
        moments=moments,
        BSorder=BSorder,
        pad_mode=pad_mode
        )
    
def input_generator_noisy_sinogram(noise_level, seed):
    return add_relative_gaussian_noise(
        sinogram, 
        rel_level=noise_level, 
        seed=seed
        )

def input_generator_noisy_fbp(noise_level, seed):
    sinogram_noisy = input_generator_noisy_sinogram(noise_level, seed)
    return reconstruct_fbp(
        sino=sinogram_noisy, 
        A=A, 
        filter_name="Ram-Lak"
        )
    

### Setup for MC

In [5]:


dx, dy = space.cell_sides           # physical grid spacing (x, y)
_, d_det = data_space.cell_sides    # (angular res (rad), detector spacing)

method_dg_fixed = partial(
    method_dg,
    xlim=(xmin, xmax),
    ylim=(ymin, ymax),
)

method_fourier_fixed = partial(
    method_fourier,
    dy=dy,
    dx=dx,
)

method_detector_fixed = partial(
    method_detector,
    A=A,
    d_det=d_det,
)

noise_levels = np.concatenate([
    np.linspace(0.01, 0.1, 10),         # low to moderate noise regime
    np.array([0.15, 0.2, 0.25, 0.3])    # high noise regime
])                                      # noise levels to test

n_reps = 20                                 # number of MC simulations
ground_truth = phantom_np                   # ground truth

# Parameter grid for method: Nodal-->Modal-->SIAC parameters
ps = [1, 3]
moments_dg_siac = [2, 4]
BSorders_dg_siac = np.arange(1, 4+1)

# Parameter grid for method: Fourier SIAC filtering in detector variable
moments_detector = [2, 4, 6]
BSorders_detector = np.arange(8, 20+1)

# Parameter grid for method: Fourier SIAC filtering as seperable fourier multiplier
moments_fourier = [4, 6]
BSorders_fourier = np.arange(3, 7+1)


**Run MC for the DG-SIAC method**

In [6]:
results_dg = run_monte_carlo_study(
    method_fn=method_dg_fixed,
    method_name="dg",
    method_param_grid={
        "p": ps,
        "moments": moments_dg_siac,
        "BSorder": BSorders_dg_siac,
    },
    noise_levels=noise_levels,
    n_reps=n_reps,
    input_generator=input_generator_noisy_fbp,
    metric_fn=compute_metrics,
    reference=ground_truth,
    base_seed=1234,
)

Noise levels: 100%|██████████| 14/14 [04:56<00:00, 21.19s/it]


**Run MC for the Fourier-SIAC method**

In [7]:
results_fourier = run_monte_carlo_study(
    method_fn=method_fourier_fixed,
    method_name="fourier",
    method_param_grid={
        "moments": moments_fourier,
        "BSorder": BSorders_fourier,
    },
    noise_levels=noise_levels,
    n_reps=n_reps,
    input_generator=input_generator_noisy_fbp,
    metric_fn=compute_metrics,
    reference=ground_truth,
    base_seed=1234,
)

Noise levels: 100%|██████████| 14/14 [00:24<00:00,  1.74s/it]


**Run MC for the Detector-SIAC method**

In [8]:
results_detector = run_monte_carlo_study(
    method_fn=method_detector_fixed,
    method_name="detector",
    method_param_grid={
        "moments": moments_detector,
        "BSorder": BSorders_detector,
    },
    noise_levels=noise_levels,
    n_reps=n_reps,
    input_generator=input_generator_noisy_sinogram,
    metric_fn=compute_metrics,
    reference=ground_truth,
    base_seed=1234,
)

Noise levels: 100%|██████████| 14/14 [03:09<00:00, 13.53s/it]


In [9]:
# Save results to csv files
results_dg.to_csv("results_dg.csv", index=False)
results_fourier.to_csv("results_fourier.csv", index=False)
results_detector.to_csv("results_detector.csv", index=False)